# Naive Bayes from Scratch — Text Classification

This notebook implements a simple Multinomial Naive Bayes classifier for text classification using pure Python. We
use a tiny, hand-crafted dataset (no external files or libraries required).

## Contents
- Tokenization (lowercasing, basic regex)
- Building a vocabulary and word counts per class
- Multinomial Naive Bayes with Laplace smoothing
- Prediction with log-probabilities (to avoid underflow)
- Simple train/test split evaluation
- Inspecting top indicative words per class

In [1]:
import re
import math
import random
from collections import Counter, defaultdict
from typing import List, Tuple, Dict, Iterable

random.seed(42)  # for reproducibility

## A Tiny Sentiment Dataset
We'll classify short movie review snippets as `pos` (positive) or `neg` (negative).

In [2]:
data: List[Tuple[str, str]] = [
    ("I loved this movie it was fantastic and fun", "pos"),
    ("What a wonderful film absolutely enjoyed it", "pos"),
    ("Great acting and a touching story", "pos"),
    ("A delightful, heartwarming experience", "pos"),
    ("Brilliant cinematography and excellent performances", "pos"),
    ("This was terrible I hated every minute", "neg"),
    ("Awful plot and boring characters", "neg"),
    ("Waste of time not recommended", "neg"),
    ("Poorly written and painfully slow", "neg"),
    ("Disappointing and forgettable", "neg"),
    ("Absolutely fantastic performances and a great soundtrack", "pos"),
    ("Mediocre at best lacked charm", "neg"),
    ("Heartfelt and beautifully made", "pos"),
    ("Not good clumsy direction", "neg"),
    ("An inspiring and joyful film", "pos"),
    ("One of the worst movies I've seen", "neg"),
    ("I smiled the whole time", "pos"),
    ("It dragged on and on", "neg"),
    ("A charming and uplifting story", "pos"),
    ("Truly dreadful experience", "neg"),
]

len(data), data[:3]

(20,
 [('I loved this movie it was fantastic and fun', 'pos'),
  ('What a wonderful film absolutely enjoyed it', 'pos'),
  ('Great acting and a touching story', 'pos')])

## Tokenization
We'll keep tokenization simple: lowercase and extract alphabetic words with apostrophes (e.g., `don't`). A small set of stopwords is optional.

In [3]:
STOPWORDS = {
    'the','a','an','and','or','of','to','it','was','is','at','on','for','this','that','i'
}

def tokenize(text: str, *, remove_stopwords: bool = False) -> List[str]:
    tokens = re.findall(r"[a-z']+", text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

# Quick sanity check
tokenize("This was TERRIFIC!!"), tokenize("Don't stop", remove_stopwords=True)

(['this', 'was', 'terrific'], ["don't", 'stop'])

## Multinomial Naive Bayes (from scratch)
Given a document represented by token counts, Naive Bayes scores each class with:

log P(class) + sum over tokens: count(token) * log P(token | class)

We use Laplace smoothing so unseen words don't produce zero probabilities.

In [4]:
class NaiveBayesClassifier:
    def __init__(self, alpha: float = 1.0, tokenizer=tokenize, remove_stopwords: bool = False):
        self.alpha = alpha
        self.tokenizer = tokenizer
        self.remove_stopwords = remove_stopwords
        # learned parameters
        self.class_priors: Dict[str, float] = {}
        self.class_word_counts: Dict[str, Counter] = {}
        self.class_total_words: Dict[str, int] = {}
        self.vocab: set = set()
        self.classes_: List[str] = []

    def fit(self, texts: Iterable[str], labels: Iterable[str]):
        # Count documents per class
        label_list = list(labels)
        text_list = list(texts)
        n_docs = len(text_list)
        class_doc_counts = Counter(label_list)
        self.classes_ = sorted(class_doc_counts.keys())
        self.class_priors = {c: class_doc_counts[c] / n_docs for c in self.classes_}

        # Initialize word counters
        self.class_word_counts = {c: Counter() for c in self.classes_}
        self.class_total_words = {c: 0 for c in self.classes_}
        self.vocab = set()

        # Accumulate word counts per class
        for text, label in zip(text_list, label_list):
            tokens = self.tokenizer(text, remove_stopwords=self.remove_stopwords)
            self.vocab.update(tokens)
            counts = Counter(tokens)
            self.class_word_counts[label].update(counts)
            self.class_total_words[label] += sum(counts.values())

        return self

    def _log_likelihood(self, token: str, label: str) -> float:
        # P(token|label) with Laplace smoothing
        V = len(self.vocab)
        count_wc = self.class_word_counts[label][token]
        total_c = self.class_total_words[label]
        return math.log((count_wc + self.alpha) / (total_c + self.alpha * V))

    def predict_one(self, text: str) -> str:
        tokens = self.tokenizer(text, remove_stopwords=self.remove_stopwords)
        counts = Counter(tokens)
        V = len(self.vocab)
        # If we encounter OOV tokens, Laplace denominator accounts via V;
        # we still use the smoothed probability for unseen words.
        best_label, best_score = None, -float('inf')
        for c in self.classes_:
            score = math.log(self.class_priors[c])
            for token, cnt in counts.items():
                # log P(token|c) multiplied by its count in the doc
                ll = self._log_likelihood(token, c) if V > 0 else -float('inf')
                score += cnt * ll
            if score > best_score:
                best_label, best_score = c, score
        return best_label

    def predict(self, texts: Iterable[str]) -> List[str]:
        return [self.predict_one(t) for t in texts]

    def score(self, texts: Iterable[str], labels: Iterable[str]) -> float:
        preds = self.predict(texts)
        y = list(labels)
        correct = sum(p == yi for p, yi in zip(preds, y))
        return correct / len(y) if y else 0.0

    def top_informative(self, n: int = 10) -> Dict[str, List[Tuple[str, float]]]:
        # Compute per-class log odds vs. the complement class (for two-class problems).
        result = {}
        if len(self.classes_) != 2:
            return result
        c1, c2 = self.classes_
        V = len(self.vocab)
        def logpw(token, c):
            count_wc = self.class_word_counts[c][token]
            total_c = self.class_total_words[c]
            return math.log((count_wc + self.alpha) / (total_c + self.alpha * V))
        scores_c1 = []
        scores_c2 = []
        for w in self.vocab:
            s1 = logpw(w, c1) - logpw(w, c2)
            s2 = logpw(w, c2) - logpw(w, c1)
            scores_c1.append((w, s1))
            scores_c2.append((w, s2))
        scores_c1.sort(key=lambda x: x[1], reverse=True)
        scores_c2.sort(key=lambda x: x[1], reverse=True)
        result[c1] = scores_c1[:n]
        result[c2] = scores_c2[:n]
        return result

# Quick smoke test on tiny inputs
clf = NaiveBayesClassifier(alpha=1.0, remove_stopwords=True)
clf.fit([t for t, y in data], [y for t, y in data])
clf.predict_one("fantastic and great performances"), clf.predict_one("boring and awful")

('pos', 'neg')

## Train/Test Split and Evaluation

In [5]:
def train_test_split(dataset: List[Tuple[str, str]], test_size: float = 0.3) -> Tuple[list, list]:
    idx = list(range(len(dataset)))
    random.shuffle(idx)
    split = int(len(idx) * (1 - test_size))
    train_idx, test_idx = idx[:split], idx[split:]
    train = [dataset[i] for i in train_idx]
    test = [dataset[i] for i in test_idx]
    return train, test

train, test = train_test_split(data, test_size=0.3)
len(train), len(test), train[:2], test[:2]

(14,
 6,
 [('Truly dreadful experience', 'neg'),
  ('This was terrible I hated every minute', 'neg')],
 [('Great acting and a touching story', 'pos'),
  ('I smiled the whole time', 'pos')])

In [7]:
# Train
clf = NaiveBayesClassifier(alpha=1.0, remove_stopwords=True)
X_train = [t for t, y in train]
y_train = [y for t, y in train]
X_test  = [t for t, y in test]
y_test  = [y for t, y in test]

clf.fit(X_train, y_train)
acc = clf.score(X_test, y_test)
print(f'Accuracy: {acc:.2f} ({sum(clf.predict(X_test)[i]==y_test[i] for i in range(len(y_test)))}/{len(y_test)})')

# Show a few predictions
for t, y in test[:5]:
    print(f'Text: {t} True: {y}  Pred: {clf.predict_one(t)}')

Accuracy: 0.67 (4/6)
Text: Great acting and a touching story True: pos  Pred: pos
Text: I smiled the whole time True: pos  Pred: neg
Text: Waste of time not recommended True: neg  Pred: neg
Text: Poorly written and painfully slow True: neg  Pred: neg
Text: I loved this movie it was fantastic and fun True: pos  Pred: pos


## Most Informative Words
For two classes, we can inspect words that most strongly indicate each class using log-odds differences.

In [8]:
info = clf.top_informative(n=8)
for label, words in info.items():
    print(f'Top words for class {label}:')
    for w, s in words:
        print(f'  {w:15s}  log-odds: {s:+.3f}')

Top words for class neg:
  lacked           log-odds: +0.638
  every            log-odds: +0.638
  direction        log-odds: +0.638
  boring           log-odds: +0.638
  hated            log-odds: +0.638
  clumsy           log-odds: +0.638
  one              log-odds: +0.638
  characters       log-odds: +0.638
Top words for class pos:
  film             log-odds: +1.154
  absolutely       log-odds: +1.154
  performances     log-odds: +1.154
  heartfelt        log-odds: +0.749
  brilliant        log-odds: +0.749
  soundtrack       log-odds: +0.749
  story            log-odds: +0.749
  joyful           log-odds: +0.749


## Try Your Own Sentences

In [ ]:
samples = [
    "absolutely wonderful and inspiring",
    "boring waste of time",
    "great soundtrack but the story was mediocre",
    "painfully slow and disappointing",
]
for s in samples:
    print(f'{s!r} -> {clf.predict_one(s)}')

## Notes and Next Steps
- The dataset here is intentionally tiny; accuracy will vary by split.
- The model is Multinomial NB (counts). For short texts, a Bernoulli variant (presence/absence) can also work.
- You can tune `alpha` (smoothing) and `remove_stopwords` to see effects.
- For real tasks, use a larger dataset and consider preprocessing like n-grams.

In [ ]:
from collections import Counter

def tokenize(text):
    return text.lower().split()

class SimpleNB:
    def fit(self, texts, labels):
        print("=== TRAINING PHASE ===")
        self.classes = set(labels)
        self.class_counts = Counter(labels)
        self.total_docs = len(labels)

        # word counts per class
        self.word_counts = {c: Counter() for c in self.classes}
        self.total_words = {c: 0 for c in self.classes}

        for text, label in zip(texts, labels):
            tokens = tokenize(text)
            self.word_counts[label].update(tokens)
            self.total_words[label] += len(tokens)
            print(f"Doc: '{text}' → Class: {label}, Tokens: {tokens}")

        print("\nClass priors:")
        for c in self.classes:
            print(f"P({c}) = {self.class_counts[c]} / {self.total_docs} = {self.class_counts[c]/self.total_docs:.2f}")

        print("\nWord counts per class:")
        for c in self.classes:
            print(f"{c}: {dict(self.word_counts[c])}")

    def predict_one(self, text):
        print("\n=== PREDICTION PHASE ===")
        tokens = tokenize(text)
        print(f"Input text: '{text}' → Tokens: {tokens}")

        scores = {}
        for c in self.classes:
            # start with prior
            score = self.class_counts[c] / self.total_docs
            print(f"\nClass: {c}, start with prior = {score:.4f}")

            for t in tokens:
                prob = (self.word_counts[c][t] + 1) / (self.total_words[c] + len(self.word_counts[c]))
                print(f"  Word '{t}': count={self.word_counts[c][t]}, "
                      f"prob = ({self.word_counts[c][t]}+1)/({self.total_words[c]}+{len(self.word_counts[c])}) = {prob:.4f}")
                score *= prob

            scores[c] = score
            print(f"Total score for class {c}: {score:.6f}")

        # pick class with max score
        best_class = max(scores, key=scores.get)
        print(f"\nFinal decision: {best_class} (scores={scores})")
        return best_class

    def predict(self, texts):
        return [self.predict_one(t) for t in texts]


data = [
    ("I love this movie", "pos"),
    ("Fantastic and amazing film", "pos"),
    ("I hate this movie", "neg"),
    ("Terrible and boring film", "neg")
]

texts, labels = zip(*data)

clf = SimpleNB()
clf.fit(texts, labels)

print("\n>>> Predicting...")
clf.predict_one("amazing movie")
clf.predict_one("boring film")


=== TRAINING PHASE ===
Doc: 'I love this movie' → Class: pos, Tokens: ['i', 'love', 'this', 'movie']
Doc: 'Fantastic and amazing film' → Class: pos, Tokens: ['fantastic', 'and', 'amazing', 'film']
Doc: 'I hate this movie' → Class: neg, Tokens: ['i', 'hate', 'this', 'movie']
Doc: 'Terrible and boring film' → Class: neg, Tokens: ['terrible', 'and', 'boring', 'film']

Class priors:
P(pos) = 2 / 4 = 0.50
P(neg) = 2 / 4 = 0.50

Word counts per class:
pos: {'i': 1, 'love': 1, 'this': 1, 'movie': 1, 'fantastic': 1, 'and': 1, 'amazing': 1, 'film': 1}
neg: {'i': 1, 'hate': 1, 'this': 1, 'movie': 1, 'terrible': 1, 'and': 1, 'boring': 1, 'film': 1}

>>> Predicting...

=== PREDICTION PHASE ===
Input text: 'amazing movie' → Tokens: ['amazing', 'movie']

Class: pos, start with prior = 0.5000
  Word 'amazing': count=1, prob = (1+1)/(8+8) = 0.1250
  Word 'movie': count=1, prob = (1+1)/(8+8) = 0.1250
Total score for class pos: 0.007812

Class: neg, start with prior = 0.5000
  Word 'amazing': count=0, p

'neg'